<a href="https://colab.research.google.com/github/mat381/2026_Intro_Python/blob/main/1.6-geospatial-vector-data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> Notebook-friendly copy of `part-I/1.6-geospatial-vector-data.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

In [ ]:
# --- environment setup (generated, not part of the lesson) ---
# Colab and Kaggle do not ship every package this notebook imports.
# Colab and Kaggle start in an empty working directory.
# This is a no-op in an environment that is already set up.
import importlib.util
import subprocess
import sys

for module, package in {"geopandas": "geopandas"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

from pathlib import Path

Path("_files").mkdir(exist_ok=True)   # the folder this notebook writes into

# 1.6) Geospatial Vector Data with geopandas

geopandas extends the pandas dataframe with a geometry column and a coordinate reference system (crs), so points, lines, and polygons can be filtered, joined, measured, and mapped with familiar syntax. This subchapter turns a table of Swiss weather-station coordinates into a GeoDataFrame, reads a file of hazard zones, measures areas and distances, reprojects between the geographic crs WGS84 (EPSG:4326) and the projected Swiss crs LV95 (EPSG:2056), and works through spatial joins, buffers, unions, dissolves, and overlays. At the end, the generated code measures distance and area in degrees instead of a projected crs — the single most common geospatial mistake.

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/geopandas_logo.png" alt="The geopandas project logo" width="500">

<em>The geopandas logo, from the project's own <a href="https://geopandas.org/en/latest/_images/geopandas_logo.png">documentation</a>.</em>

**🎯 Learning objectives**

- Build a GeoDataFrame from a table of coordinates, and tell a GeoDataFrame and a GeoSeries from their pandas counterparts.
- Read and write shp, geojson, and gpkg, and name the vector geometry types and their interior, boundary, and exterior.
- Compute area, length, distance, boundary, and centroid, and do it only in a projected (metric) crs.
- Inspect a crs and reproject with to_crs between WGS84 (EPSG:4326) and LV95 (EPSG:2056), and plot both.
- Apply spatial predicates and spatial joins.
- Combine geometries with buffer, union_all, convex_hull, envelope, dissolve, and overlay.

## 1.6.1 Installing geopandas

geopandas is part of this book's environment, and hosted notebook services ship it as well; elsewhere, `pip install geopandas` installs it together with its dependencies. It extends pandas to geospatial data by combining:

- pandas, for data analysis;
- shapely, for geometries and set-theoretic operations on planar features;
- pyogrio (or fiona, in older installations), for reading and writing files;
- pyproj, for coordinate reference systems;
- matplotlib, for plotting.

In [ ]:
%matplotlib inline
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

## 1.6.2 geopandas Data Structure

A GeoDataFrame differs from a DataFrame in one respect: it has a `geometry` column holding shapely objects — points, lines, polygons, multipolygons — together with a crs that says how those coordinates map to the Earth. The most common way to get one is from a table that already holds a longitude and a latitude column, such as a csv of station positions: `gpd.points_from_xy` turns the two columns into point geometries.

In [ ]:
stations_df = pd.DataFrame({
    "name": ["Basel", "Zurich", "Bern", "Lugano"],
    "lon": [7.59, 8.54, 7.44, 8.95],
    "lat": [47.56, 47.37, 46.95, 46.00],
})
stations = gpd.GeoDataFrame(
    stations_df,
    geometry=gpd.points_from_xy(stations_df["lon"], stations_df["lat"]),
    crs="EPSG:4326",   # WGS84 longitude/latitude in degrees
)
stations

The core data structure is `geopandas.GeoDataFrame`, a subclass of `pandas.DataFrame`, so every DataFrame method from subchapter 1.5 still works on it.

In [ ]:
type(stations)

Its geometry column is a `geopandas.GeoSeries`, a subclass of `pandas.Series` that can hold any geometry type and carries the crs. The other columns stay ordinary Series of numbers, text, or booleans, which is how a GeoDataFrame keeps traditional data and geometries side by side.

In [ ]:
print(type(stations.geometry))
print(type(stations["name"]))
print("crs:", stations.crs.to_epsg(), "| geometry type:", stations.geom_type.iloc[0])

In [ ]:
# a quick look at the geometries on a map
stations.plot()
plt.show()

## 1.6.3 Read and Write Files

### Reading files

A file that holds both data and geometry — a GeoPackage, a GeoJSON, or a shapefile — is read with `gpd.read_file`, which detects the format and returns a GeoDataFrame. The cell below writes an example file of two flood-hazard zones, stored in LV95 (EPSG:2056), the official [Swiss coordinate system](https://www.swisstopo.admin.ch/en/the-swiss-coordinates-system), whose unit is the metre.

In [ ]:
# --- generate an example data file (uses tools covered later; just setup here) ---
from pathlib import Path
from shapely.geometry import Polygon

Path("_files").mkdir(exist_ok=True)
gpd.GeoDataFrame(
    {"hazard": ["flood", "flood"], "zone_id": ["north-west", "north-east"]},
    geometry=[Polygon([(2_590_000, 1_230_000), (2_650_000, 1_230_000),
                       (2_650_000, 1_290_000), (2_590_000, 1_290_000)]),
              Polygon([(2_650_000, 1_230_000), (2_710_000, 1_230_000),
                       (2_710_000, 1_290_000), (2_650_000, 1_290_000)])],
    crs="EPSG:2056",
).to_file("_files/hazard_zones.gpkg", driver="GPKG")

In [ ]:
zones = gpd.read_file("_files/hazard_zones.gpkg")
zones

### Vector geometries

shapely implements three fundamental types of geometric object. Each is defined by an *interior*, a *boundary*, and an *exterior*:

- **Point** — a single pair of coordinates, such as a weather station or a tree. Interior: exactly that point. Boundary: no points at all. Exterior: every other point.
- **LineString** — at least two connected points, such as a road or a stream; a **LinearRing** is a closed one. Interior: the points along its length. Boundary: its two end points. Exterior: every other point.
- **Polygon** — at least three points joined into a closed ring, such as a lake or a country. Interior: the area inside. Boundary: one or more rings (the outline, plus any holes). Exterior: every other point.

Each type has a collection form — **MultiPoint**, **MultiLineString**, **MultiPolygon** — that holds several of them in one row. A country with offshore islands is a MultiPolygon, because no single ring can enclose it. Spatial predicates are defined through these three parts: `within`, for example, means that no point of one geometry lies in the exterior of the other.

In [ ]:
# the geometry column, and the type of one geometry in it
print(zones.geometry.head(3))
print(type(zones.geometry.iloc[0]))

In [ ]:
from shapely.geometry import Point, LineString, MultiPolygon

point = Point(7.59, 47.56)
line = LineString([(0, 0), (1, 1), (2, 1)])
square = Polygon([(0, 0), (1, 0), (1, 1), (0, 1)])
two_squares = MultiPolygon([square, Polygon([(2, 0), (3, 0), (3, 1), (2, 1)])])

for geom in [point, line, square, two_squares]:
    print(f"{geom.geom_type:12s} boundary: {geom.boundary}")

In [ ]:
# a quick look at the zones on a map
zones.plot(column="zone_id", edgecolor="k", legend=True)
plt.show()

### Writing files

geopandas writes shapefiles (.shp), GeoJSON (.geojson), and GeoPackage (.gpkg) with `to_file`, choosing the format from the file extension or from the `driver` keyword. GeoPackage is a single-file, modern default; a shapefile is really several files and has column-name and size limits.

In [ ]:
from pathlib import Path

Path("_files").mkdir(exist_ok=True)
stations.to_file("_files/stations.geojson", driver="GeoJSON")
stations.to_file("_files/stations.gpkg", driver="GPKG")

reread = gpd.read_file("_files/stations.gpkg")
print("reread shape:", reread.shape, "| crs preserved:", reread.crs.to_epsg())

In [ ]:
# a shapefile round-trip, written and read the same way as GeoJSON/GeoPackage above
stations.to_file("_files/stations.shp")   # driver inferred from the .shp extension
reread_shp = gpd.read_file("_files/stations.shp")
print("reread shape:", reread_shp.shape, "| crs preserved:", reread_shp.crs.to_epsg())

# a shapefile is not one file: this write also creates _files/stations.shx, .dbf, .prj (and more)
print("shapefile sidecar files:", sorted(p.name for p in Path("_files").glob("stations.*")))

## 1.6.4 Attributes and Methods

### Area, length, and distance

`.area` returns the area of each polygon as a Series, `.length` the length of each line or outline, and `.distance` the shortest distance from each geometry to another one. All three are in the units of the crs. The zones are in LV95, whose unit is the metre, so the numbers below are square metres and metres; in a geographic crs the same calls return square degrees and degrees, which are not areas or lengths at all.

In [ ]:
zones["area_km2"] = zones.area / 1e6
zones["perimeter_km"] = zones.length / 1e3
zones[["zone_id", "area_km2", "perimeter_km"]]

### Boundary and centroid

`.boundary` returns the outline of each polygon as a line, and `.centroid` a single representative point — useful for labelling a zone, or for measuring to it without carrying the whole shape around.

In [ ]:
zone_outlines = zones.boundary
zone_centres = zones.centroid
print(zone_outlines.geom_type.tolist())
print(zone_centres)

In [ ]:
# distance from each zone's centre to the first zone's centre, in metres
first_centre = zone_centres.iloc[0]
print((zone_centres.distance(first_centre) / 1000).tolist(), "km")

### Projection

A crs tells Python how the coordinates of a geometry relate to places on the Earth; without one they are numbers in an arbitrary plane. A map projection — a projected crs — transforms longitude and latitude onto a flat surface, usually measured in metres instead of degrees. Layers stored in different crs do not line up, so a common first step is to reproject one layer into the crs of the other: only then can they be related, for instance by asking which points fall inside which polygon.

A crs is usually named by its EPSG code, a number in the [EPSG registry](https://epsg.org/) (named after the European Petroleum Survey Group, which started it). `.crs` shows a layer's crs, and `to_crs` reprojects it.

In [ ]:
stations_lv95 = stations.to_crs(2056)   # reproject to metric Swiss coordinates

print("before:", stations.crs.to_epsg(), "->", "after:", stations_lv95.crs.to_epsg())
print("Basel WGS84 (deg):", round(stations.geometry.x.iloc[0], 3), round(stations.geometry.y.iloc[0], 3))
print("Basel LV95 (m):  ", round(stations_lv95.geometry.x.iloc[0], 1), round(stations_lv95.geometry.y.iloc[0], 1))

In [ ]:
zones_wgs84 = zones.to_crs(4326)     # the zones, from LV95 metres to WGS84 degrees
print("before:", zones.crs.to_epsg(), "->", "after:", zones_wgs84.crs.to_epsg())
print("zone bounds in degrees:", zones_wgs84.total_bounds.round(3))

**🧠 Computational-thinking fundamental: coordinates are meaningless without a crs**

A geometry is only a list of numbers until a crs says what those numbers mean. Degrees of longitude and latitude are angles, not distances — one degree of longitude spans about 111 km at the equator but shrinks to zero at the poles — so any area, length, or distance computed in a geographic crs is nonsense. Reproject to a projected crs with metric units (here LV95, EPSG:2056) before measuring anything. Pick the crs based on what the numbers should mean physically — the map will still look fine with the wrong one, but every distance and area computed from it will be wrong.

### Plot

geopandas plots the active geometry on a matplotlib axes with `.plot()`, and calling `.plot()` again on the same axes adds a layer. Drawing the same two layers in both crs shows what a projection does: the zones are rectangles in LV95 but not in WGS84, because the two systems lay the Earth out on the plane differently.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
zones_wgs84.plot(ax=axs[0], facecolor="lightgray", edgecolor="k")
stations.plot(ax=axs[0], color="tab:red")
axs[0].set_title("WGS84 (EPSG:4326), degrees")
zones.plot(ax=axs[1], facecolor="lightblue", edgecolor="k")
stations_lv95.plot(ax=axs[1], color="tab:red")
axs[1].set_title("LV95 (EPSG:2056), metres")
fig.tight_layout()
plt.show()

## 1.6.5 Spatial Relationships and Operations

Spatial predicates (`within`, `intersects`, `contains`) test topological relationships between geometries. A spatial join (`sjoin`) attaches the attributes of one layer to another by location rather than by a shared key. Both need the two layers in the same crs, so the stations are joined here in LV95.

In [ ]:
# which stations fall within a hazard zone?
in_hazard = gpd.sjoin(stations_lv95, zones, predicate="within", how="inner")
print("stations in a hazard zone:", in_hazard["name"].tolist())

# the same relationship as a predicate: does each zone contain Basel?
print(zones.contains(stations_lv95.geometry.iloc[0]).tolist())

The cell below flags each station with `.isin()`, which checks whether a value appears in a given list or Series — here, whether a station's name is among the names the spatial join found inside a hazard zone — and maps the result.

In [ ]:
flagged = stations.assign(in_hazard=stations["name"].isin(in_hazard["name"]))

fig, ax = plt.subplots(figsize=(5, 5))
zones_wgs84.boundary.plot(ax=ax, color="tab:red", linewidth=1)
flagged.plot(ax=ax, column="in_hazard", categorical=True, legend=True, markersize=40)
ax.set_xlabel("longitude (°E)")
ax.set_ylabel("latitude (°N)")
ax.set_title("stations and hazard zones (EPSG:4326)")
plt.show()

## 1.6.6 Geometric Manipulations

geopandas makes shapely's geometric manipulations available on a whole layer at once — among them [`buffer`, `boundary`, `convex_hull`, `envelope`, and `union_all`](https://geopandas.org/en/stable/docs/user_guide/geometric_manipulations.html). `buffer` grows a geometry by a distance in crs units, so project first; `union_all` collapses every geometry in a layer into one.

In [ ]:
# a 40 km radius around each station (metric crs required)
buffers = stations_lv95.buffer(40_000)
print("one buffer:", round(buffers.area.iloc[0] / 1e6), "km^2")

# union_all merges the four circles; where two overlap, the shared area is counted once
all_buffers = buffers.union_all()
print(all_buffers.geom_type, round(all_buffers.area / 1e6), "km^2",
      "| sum of the four:", round(buffers.area.sum() / 1e6), "km^2")

In [ ]:
# the smallest convex polygon, and the smallest axis-aligned rectangle, around all four stations
all_stations = stations_lv95.union_all()
print("convex hull:", round(all_stations.convex_hull.area / 1e6), "km^2")
print("envelope:   ", round(all_stations.envelope.area / 1e6), "km^2")

`dissolve` is the attribute-driven version of `union_all`: it merges the geometries that share a value in a given column, much as `groupby` aggregates rows.

In [ ]:
dissolved = zones.dissolve(by="hazard")
print("polygons after dissolve:", len(dissolved))
print("dissolved hazard-zone centroid (m):", dissolved.centroid.iloc[0])

### Set operations with overlay

With several polygon layers, you often need the shapes where they overlap, or where they do not. These are the set operations — intersection, union, difference — and [`overlay`](https://geopandas.org/en/stable/docs/user_guide/set_operations.html) applies them to two layers. `how="intersection"` keeps only the area covered by both.

In [ ]:
basel_buffer = gpd.GeoDataFrame({"name": ["Basel"]}, geometry=[buffers.iloc[0]], crs=2056)
clip = gpd.overlay(zones, basel_buffer, how="intersection")
print("overlay produced", len(clip), "piece(s), area", round(clip.area.sum() / 1e6, 1), "km^2")

ax = clip.plot(column="zone_id", cmap="tab10")
zones.plot(ax=ax, facecolor="none", edgecolor="k")
basel_buffer.plot(ax=ax, facecolor="none", edgecolor="k", linestyle="--")
plt.show()

**ℹ️ Quick exercise: distance between two stations**

Reproject the stations to EPSG:2056 and compute the distance between Basel and Bern in kilometres.


<details>
<summary><b>✅ Solution</b></summary>

```python
lv95 = stations.to_crs(2056)
basel = lv95.loc[lv95["name"] == "Basel", "geometry"].iloc[0]
bern = lv95.loc[lv95["name"] == "Bern", "geometry"].iloc[0]
print(round(basel.distance(bern) / 1000, 1), "km")
```

</details>

## *When generated code lies: measuring in degrees*

Asked for the distance between two stations, an assistant calls `.distance()` on the data as loaded — in EPSG:4326. The result is a number of *degrees*, not metres. geopandas even warns, but the value looks plausible and flows on into whatever comes next.

In [ ]:
# the "distance between two stations" as an assistant might compute it
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    dist_deg = stations.geometry.distance(stations.geometry.iloc[2])   # Basel..Bern etc., in EPSG:4326
    area_deg = gpd.GeoSeries(
        [Polygon([(7, 47), (8, 47), (8, 48), (7, 48)])], crs="EPSG:4326").area

print("Basel->Bern distance in degrees (wrong):", round(dist_deg.iloc[0], 4))
print("1x1 degree box area in deg^2 (wrong):", round(area_deg.iloc[0], 4))
print("geopandas warned about a geographic crs:",
      any("geographic CRS" in str(warning.message) for warning in caught))

print(f'\n{caught[0].message}')

**⚠️ Diagnosis: measure in a projected crs**

The distance came back as roughly 0.6 "degrees" and the box as 1.0 "square degree" — quantities with no physical meaning, because subtracting angles does not give a length. geopandas raised a UserWarning, but a warning is easy to miss and the pipeline kept running. The fix is to reproject to a metric crs (EPSG:2056) and then measure; the same call now returns metres and square metres.

In [ ]:
dist_m = stations_lv95.geometry.distance(stations_lv95.geometry.iloc[2])
area_m2 = gpd.GeoSeries(
    [Polygon([(7, 47), (8, 47), (8, 48), (7, 48)])], crs="EPSG:4326").to_crs(2056).area

print("Basel->Bern distance:", round(dist_m.iloc[0] / 1000, 1), "km")
print("1x1 degree box area:", round(area_m2.iloc[0] / 1e6, 0), "km^2")

<details>
<summary><b>🔍 Going deeper: raster data with rioxarray</b></summary>

Vector data has geometries; raster data has a grid of cells (a digital elevation model, satellite imagery). rioxarray adds a crs and geotransform to an xarray DataArray.

```python
import rioxarray
dem = rioxarray.open_rasterio("dem.tif")      # dims (band, y, x), with a crs
dem_lv95 = dem.rio.reproject("EPSG:2056")
```

The same crs discipline applies: reproject before measuring.

</details>

<details>
<summary><b>🔍 Going deeper: zonal statistics</b></summary>

Zonal statistics summarise raster values within vector polygons — for example the mean elevation of each catchment.

```python
from rasterstats import zonal_stats
stats = zonal_stats("catchments.gpkg", "dem.tif", stats=["mean", "max"])
```

Zonal statistics is the usual way raster and vector data meet in a workflow: the vector polygon defines a zone, the raster supplies the values inside it.

</details>

<details>
<summary><b>🔍 Going deeper: slope and aspect from a DEM</b></summary>

Terrain derivatives come from the spatial gradient of an elevation grid. With a DEM as an array of elevations and a grid spacing `dx`, `dy`:

```python
import numpy as np
dz_dy, dz_dx = np.gradient(elevation, dy, dx)
slope = np.arctan(np.hypot(dz_dx, dz_dy))     # radians
aspect = np.arctan2(-dz_dy, dz_dx)            # direction of steepest descent
```

Dedicated tools (richdem, gdaldem) handle edge effects and units more carefully.

</details>

<details>
<summary><b>🔍 Going deeper: interactive web maps with folium</b></summary>

`GeoDataFrame.explore()` builds a leaflet web map (via folium) for interactive inspection in a notebook.

```python
stations.explore(column="name", tiles="OpenStreetMap")
```

Useful for exploration; static matplotlib maps remain better for print figures.

</details>

**📌 Takeaways**

- A GeoDataFrame is a DataFrame with a geometry column (a GeoSeries) and a crs; the crs gives the coordinates physical meaning.
- `gpd.points_from_xy` turns longitude and latitude columns into point geometries.
- Read and write shp, geojson, and gpkg with `read_file`/`to_file`; prefer GeoPackage to shapefile.
- Points, lines, and polygons each have an interior, a boundary, and an exterior; a MultiPolygon holds several polygons in one row.
- Inspect the crs with `.crs` and reproject with `to_crs`; measure area, length, and distance only in a projected metric crs (here EPSG:2056), never in degrees.
- Use spatial predicates and `sjoin` to relate layers by location, with both layers in the same crs.
- Combine geometries with `buffer` (project first), `union_all`, `dissolve` (merge by attribute), and `overlay` (set operations); `.boundary`, `.centroid`, `convex_hull`, and `envelope` derive new geometries from old ones.

## Summary

| Concept | Rule to remember |
|---|---|
| GeoDataFrame | A DataFrame with a geometry column (a GeoSeries) and a crs; build one from coordinates with `points_from_xy`. |
| Formats | `read_file`/`to_file` handle shp, geojson, and gpkg; prefer GeoPackage to shapefile. |
| Geometries | Point, LineString, Polygon, and their Multi- forms, each with an interior, boundary, and exterior. |
| The crs | What gives coordinates physical meaning; read it with `.crs`, change it with `to_crs`. |
| Measuring | Area, length, and distance only in a projected metric crs — never in degrees. |
| Relating layers | Spatial predicates and `sjoin` join two layers by location, in one crs. |
| Combining geometries | `buffer` (project first), `union_all`, `dissolve` (merge by attribute), `overlay` (set operations). |

## Resources

- [Introduction to GeoPandas](https://geopandas.org/en/stable/getting_started/introduction.html) — geopandas' own walk through its data structures, reading, and plotting.
- [GeoPandas — Managing projections](https://geopandas.org/en/stable/docs/user_guide/projections.html) — setting and transforming coordinate reference systems, including why measurements require a projected crs.
- [Geopandas: an introduction](https://autogis-site.readthedocs.io/en/latest/lessons/lesson-2/geopandas-an-introduction.html) — from *Automating GIS Processes*; reading, writing, and measuring vector data.
- [The Shapely User Manual](https://shapely.readthedocs.io/en/stable/manual.html) — the geometry types, their interior, boundary, and exterior, and the predicates defined on them.
- [Use Data for Earth and Environmental Science in Open Source Python](https://www.earthdatascience.org/courses/use-data-open-source-python/) — vector and raster data in environmental applications.
- [Geospatial Analysis with Python and R](https://kodu.ut.ee/~kmoch/geopython2020/index.html) — a course covering geopandas, projections, and spatial operations.
- [GeoPandas basics: maps, projections, and spatial joins](https://realpython.com/geopandas/) — a worked introduction to GeoDataFrames, crs handling, spatial joins, and plotting.